In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch

print("Python/Torch environment")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM GiB:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3)
)

assert torch.cuda.is_available()
assert "T4" in torch.cuda.get_device_name(0)
assert torch.__version__ == "2.10.0+cu128"
assert torch.version.cuda == "12.8"

print("\n✅ Base GPU environment correct")

Python/Torch environment
Torch: 2.10.0+cu128
CUDA available: True
CUDA: 12.8
GPU: Tesla T4
VRAM GiB: 14.562

✅ Base GPU environment correct


In [2]:
!python -m pip install -q \
    "transformers==5.0.0" \
    "accelerate==1.13.0" \
    "bitsandbytes==0.50.2" \
    "peft==0.19.1" \
    "trl==1.13.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.4 MB/s eta 0:00:00


In [3]:
!python -c "import torch,transformers,accelerate,bitsandbytes,peft,trl; \
print('Torch:',torch.__version__); \
print('CUDA:',torch.version.cuda); \
print('Transformers:',transformers.__version__); \
print('Accelerate:',accelerate.__version__); \
print('bitsandbytes:',bitsandbytes.__version__); \
print('PEFT:',peft.__version__); \
print('TRL:',trl.__version__); \
print('GPU:',torch.cuda.get_device_name(0))"

Torch: 2.10.0+cu128
CUDA: 12.8
Transformers: 5.0.0
Accelerate: 1.13.0
bitsandbytes: 0.50.2
PEFT: 0.19.1
TRL: 1.13.0
GPU: Tesla T4


In [5]:
from pathlib import Path
import shutil
import zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/localsql")

if WORK.exists():
    shutil.rmtree(WORK)

# -------------------------------------------------
# Locate Phase 5C source
# -------------------------------------------------

runner_matches = [
    p for p in INPUT.rglob("run_qlora_full_training.py")
    if "phase5" in str(p).lower()
]

if runner_matches:
    print("Direct Phase 5C source candidates:")
    for p in runner_matches:
        print(" ", p)

    assert len(runner_matches) == 1, (
        f"Expected 1 Phase 5C runner, found {len(runner_matches)}"
    )

    SRC = runner_matches[0].parent.parent
    shutil.copytree(SRC, WORK)

else:
    phase5c_zips = [
        p for p in INPUT.rglob("*.zip")
        if "phase5c" in p.name.lower()
    ]

    print("Phase 5C ZIP candidates:")
    for p in phase5c_zips:
        print(" ", p)

    assert len(phase5c_zips) == 1, (
        f"Expected 1 Phase 5C ZIP, found {len(phase5c_zips)}"
    )

    with zipfile.ZipFile(phase5c_zips[0], "r") as z:
        z.extractall(WORK)

# -------------------------------------------------
# Locate candidate dataset
# -------------------------------------------------

candidate_dirs = []

for train_file in INPUT.rglob("train.jsonl"):
    parent = train_file.parent

    if (
        "candidate" in str(parent).lower()
        and (parent / "validation.jsonl").exists()
        and (parent / "policy.json").exists()
    ):
        candidate_dirs.append(parent)

print("\nCandidate directories:")
for p in candidate_dirs:
    print(" ", p)

assert len(candidate_dirs) == 1, (
    f"Expected exactly one candidate dataset, found {len(candidate_dirs)}"
)

CANDIDATE_SRC = candidate_dirs[0]

candidate_dest = WORK / "data/processed_phase5_candidate"
candidate_dest.mkdir(parents=True, exist_ok=True)

for name in ["train.jsonl", "validation.jsonl", "policy.json"]:
    shutil.copy2(
        CANDIDATE_SRC / name,
        candidate_dest / name,
    )

print("\n--- SETUP RESULT ---")
print("Working repo:", WORK)
print("Candidate source:", CANDIDATE_SRC)
print(
    "Full-training runner:",
    (WORK / "scripts/run_qlora_full_training.py").exists()
)
print(
    "Train:",
    (candidate_dest / "train.jsonl").exists()
)
print(
    "Validation:",
    (candidate_dest / "validation.jsonl").exists()
)
print(
    "Policy:",
    (candidate_dest / "policy.json").exists()
)

assert (WORK / "scripts/run_qlora_full_training.py").exists()
assert (candidate_dest / "train.jsonl").exists()
assert (candidate_dest / "validation.jsonl").exists()
assert (candidate_dest / "policy.json").exists()

print("\n✅ Source + candidate dataset ready")

Direct Phase 5C source candidates:
  /kaggle/input/datasets/hassanch6138/localsql-phase5c-src/scripts/run_qlora_full_training.py

Candidate directories:
  /kaggle/input/datasets/hassanch6138/localsql-phase5-candidate

--- SETUP RESULT ---
Working repo: /kaggle/working/localsql
Candidate source: /kaggle/input/datasets/hassanch6138/localsql-phase5-candidate
Full-training runner: True
Train: True
Validation: True
Policy: True

✅ Source + candidate dataset ready


In [6]:
from pathlib import Path
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

train_path = Path(
    "/kaggle/working/localsql/"
    "data/processed_phase5_candidate/train.jsonl"
)

val_path = Path(
    "/kaggle/working/localsql/"
    "data/processed_phase5_candidate/validation.jsonl"
)

EXPECTED_TRAIN = (
    "e23a97ea746cef24b17f6bea8dc8440ab96313798837033ec76af9ca79830196"
)

EXPECTED_VAL = (
    "441194a1ca027cf0e2b6d142879737d78d7b2459e0a4f28344ef7a55ddf48e68"
)

train_hash = sha256(train_path)
val_hash = sha256(val_path)

train_count = sum(
    1 for line in train_path.open(encoding="utf-8")
    if line.strip()
)

val_count = sum(
    1 for line in val_path.open(encoding="utf-8")
    if line.strip()
)

print("Train examples:", train_count)
print("Train SHA:", train_hash)

print("\nValidation examples:", val_count)
print("Validation SHA:", val_hash)

assert train_count == 6067
assert val_count == 534
assert train_hash == EXPECTED_TRAIN
assert val_hash == EXPECTED_VAL

print("\n✅ Canonical Phase 5 candidate dataset verified")

Train examples: 6067
Train SHA: e23a97ea746cef24b17f6bea8dc8440ab96313798837033ec76af9ca79830196

Validation examples: 534
Validation SHA: 441194a1ca027cf0e2b6d142879737d78d7b2459e0a4f28344ef7a55ddf48e68

✅ Canonical Phase 5 candidate dataset verified


In [7]:
runner = Path(
    "/kaggle/working/localsql/scripts/run_qlora_full_training.py"
)

text = runner.read_text(encoding="utf-8")

assert "stop-after-global-step" in text
assert "num_train_epochs" in text

print("✅ Phase 5C full-training runner found")
print("✅ stop-after-global-step support found")
print("✅ canonical epoch-based training support found")

✅ Phase 5C full-training runner found
✅ stop-after-global-step support found
✅ canonical epoch-based training support found


In [8]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [9]:
!python -m pip install -q -e . --no-deps

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for localsql (pyproject.toml) ... done


In [10]:
SOURCE_REV = "e173c235dcfee41868010de68db471616c92db36"

print(SOURCE_REV)

e173c235dcfee41868010de68db471616c92db36


In [11]:
!python scripts/run_qlora_full_training.py \
    --run-id phase5c-dry-run \
    --save-steps 50 \
    --save-total-limit 3 \
    --stop-after-global-step 300 \
    --source-revision {SOURCE_REV} \
    --dry-run

Canonical training file: /kaggle/working/localsql/data/processed_phase5_candidate/train.jsonl (6067 examples)
Expected optimizer steps/epoch: 759
Expected total optimizer steps (2 epochs): 1518
Dry run OK -- no model loaded, no CUDA required.


In [12]:
!python scripts/run_qlora_full_training.py \
    --run-id phase5c-boundary-test \
    --save-steps 1 \
    --save-total-limit 2 \
    --stop-after-global-step 2 \
    --source-revision {SOURCE_REV}

Canonical schedule: 759 steps/epoch x 2 epochs = 1518 total optimizer steps.
Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.30MB/s]
tokenizer_config.json: 9.38kB [00:00, 18.8MB/s]
vocab.json: 2.78MB [00:00, 41.0MB/s]
merges.txt: 1.67MB [00:00, 103MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 19.5MB/s]
model.safetensors.index.json: 32.8kB [00:00, 70.5MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:35<00:00, 11.98s/it]
Download complete: 100%|████████████████████| 8.04G/8.04G [00:36<00:00, 223MB/s]
Loading weights: 100%|█| 398/398 [00:02<00:00, 143.75it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 238/238 [00:00<00:00, 1.12MB/s]
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 6067 example

In [13]:
from pathlib import Path
import json

test_run = Path("data/runs/phase5c-boundary-test")

summary_path = test_run / "summary.json"
config_path = test_run / "run_config.json"
checkpoint_2 = test_run / "checkpoint/checkpoint-2"

assert summary_path.exists()
assert config_path.exists()

summary = json.loads(
    summary_path.read_text(encoding="utf-8")
)

run_config = json.loads(
    config_path.read_text(encoding="utf-8")
)

print("SUMMARY")
print(json.dumps(summary, indent=2))

print("\nRUN CONFIG")
print(json.dumps(run_config, indent=2))

print("\ncheckpoint-2 exists:", checkpoint_2.exists())

assert checkpoint_2.exists()

print("\n✅ Boundary checkpoint exists")

SUMMARY
{
  "run_id": "phase5c-boundary-test",
  "run_type": "phase5c_canonical_full_training",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "resolved_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "trainable_param_count": 33030144,
  "total_param_count": 2238840320,
  "optimization": {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0001,
    "warmup_ratio": 0.05,
    "gradient_checkpointing": true,
    "optim": "paged_adamw_8bit",
    "seed": 42
  },
  "max_seq_length": 4096,
  "completion_only_loss": true,
  "ca

In [ ]:
# ACTUAL TRAINING START

In [ ]:
!python scripts/run_qlora_full_training.py \
    --run-id phase5c-session-1 \
    --save-steps 50 \
    --save-total-limit 3 \
    --stop-after-global-step 300 \
    --source-revision {SOURCE_REV}

Canonical schedule: 759 steps/epoch x 2 epochs = 1518 total optimizer steps.
Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
Loading weights: 100%|█| 398/398 [00:02<00:00, 147.60it/s, Materializing param=m
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 6067 examples ...
  usable: 6067  skipped (exceeds max_seq_length=4096): 0
Training (num_train_epochs=2, stop_after_global_step=300, resume_from_checkpoint=None) ...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '1.468', 'grad_norm': '6.982', 'learning_rate': '0', 'epoch': '0.001319'}
{'loss': '1.444', 'grad_norm': '4.743', 'learning_rate': '1.316e-06', 'epoch': '0.002637'}
{'loss': '1.365', 'grad_norm': '6.083', 'learning_rate': '2.632e-06', 'epoch': '0.003956'}
{'loss': '1.739', 'grad_norm': '6.684', 'learning_rate': '3.947e-06', 'epoch':